# 09 · Final-model results and error analysis

Load Notebook 08's saved best checkpoints and evaluate both fixed models on the public test splits. No training or threshold adjustment is performed.

The following completed cloud results were transcribed from the original Notebook 08 output:

| Input | Split | Epoch | Image-mean water IoU | Global water IoU | F1 | Precision | Recall |
|---|---|---:|---:|---:|---:|---:|---:|
| RGB | Test | 18 | 0.2028 | 0.3698 | 0.5399 | 0.4374 | 0.7052 |
| RGB+NIR | Test | 18 | 0.5150 | 0.7247 | 0.8404 | 0.7507 | 0.9544 |
| RGB | Bolivia | 18 | 0.3337 | 0.4594 | 0.6296 | 0.4761 | 0.9293 |
| RGB+NIR | Bolivia | 18 | 0.5793 | 0.7908 | 0.8832 | 0.8263 | 0.9484 |

The cells below reproduce evaluation from the selected cloud checkpoints and generate image-level and event-group summaries.

## 1. Select the completed run

Set `RUN_NAME` to a `worldfloods_unet_*` directory containing both `best.pt` files. An empty value chooses the latest compatible run; verify the printed path and epochs.

Evaluate full images with checkpoint preprocessing and the original argmax rule. Save analysis in a new timestamped directory. Fixed-index galleries use six test images and three Bolivia images; separate error galleries show the three lowest NIR IoUs in each split.

In [ ]:
from pathlib import Path
from datetime import datetime
import json,csv
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import display, Markdown
import torch
from torch import nn
from torch.nn import functional as F
ROOT=Path.cwd() if (Path.cwd()/"data").exists() else Path.cwd().parent
RUN_NAME=""  # Select the completed worldfloods_unet_... run if needed.
candidates=sorted(p for p in (ROOT/"runs").glob("worldfloods_unet_*")
                  if all((p/name/"best.pt").exists() for name in ["rgb","rgb_nir"]))
RUN=ROOT/"runs"/RUN_NAME if RUN_NAME else candidates[-1]
OUT=RUN/datetime.now().strftime("analysis_%Y%m%d_%H%M%S_%f")
OUT.mkdir(parents=True)
BASE=ROOT/"data/sen1floods11/v1.1"
HAND=BASE/"data/flood_events/HandLabeled"
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
SPLITS={}
for split in ["test","bolivia"]:
    with (BASE/f"splits/flood_handlabeled/flood_{split}_data.csv").open(newline="") as f:
        SPLITS[split]=list(csv.reader(f))
EXPERIMENTS={"rgb":["B4","B3","B2"],"rgb_nir":["B4","B3","B2","B8"]}
print("Checkpoint run:",RUN,"\nAnalysis output:",OUT,"\nDevice:",device)
def save_csv(path,rows):
    with path.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader();w.writerows(rows)
def table(rows,columns):
    def fmt(v): return f"{v:.4f}" if isinstance(v,(float,np.floating)) else str(v)
    display(Markdown("\n".join(["|"+"|".join(columns)+"|","|"+"---|"*len(columns)]+
        ["|"+"|".join(fmt(r[k]) for k in columns)+"|" for r in rows])))

## 2. Model and preprocessing

Reconstruct the 64/128/256/512 U-Net with three pools and a bilinear decoder. Use each checkpoint's training means and standard deviations. Evaluation runs under `eval()` and `inference_mode()`.

In [ ]:
import torch
from torch import nn

class DoubleConv(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.encoders = nn.ModuleList([
            DoubleConv(in_channels, 64), DoubleConv(64, 128),
            DoubleConv(128, 256), DoubleConv(256, 512),
        ])
        self.pool = nn.MaxPool2d(2)
        self.decoders = nn.ModuleList([
            DoubleConv(512 + 256, 256), DoubleConv(256 + 128, 128),
            DoubleConv(128 + 64, 64),
        ])
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        skips = []
        for i, encoder in enumerate(self.encoders):
            if i:
                x = self.pool(x)
            x = encoder(x)
            skips.append(x)
        for decoder, skip in zip(self.decoders, reversed(skips[:-1])):
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=True)
            x = decoder(torch.cat([x, skip], dim=1))
        return self.head(x)




def read_sample(row):
    image_name,label_name=row
    path=HAND/"S2Hand"/image_name.replace("_S1Hand","_S2Hand")
    with rasterio.open(path) as f:
        indices=[f.descriptions.index(b)+1 for b in ["B4","B3","B2","B8"]]
        x=f.read(indices,masked=True).astype("float32").filled(np.nan)/10000.0
        profile=f.profile.copy()
    with rasterio.open(HAND/"LabelHand"/label_name) as f: y=f.read(1).astype("int64")
    y[~np.isfinite(x).all(axis=0)]=-1
    return x,y,profile

def safe_div(a,b): return float(a/b) if b else float("nan")
def pixel_counts(pred,y):
    valid=y!=-1
    return np.bincount((2*y[valid]+pred[valid]).ravel(),minlength=4).astype(np.int64)
def scores(counts):
    tn,fp,fn,tp=map(int,counts)
    return dict(water_iou=safe_div(tp,tp+fp+fn), f1=safe_div(2*tp,2*tp+fp+fn),
                precision=safe_div(tp,tp+fp), recall=safe_div(tp,tp+fn),
                true_water_fraction=safe_div(tp+fn,tn+fp+fn+tp),
                predicted_water_fraction=safe_div(tp+fp,tn+fp+fn+tp))

## 3. Full test inference

Save each image's TP/FP/FN/TN counts, valid-pixel count, water IoU and water fractions. Keep all-ignored images in the table with undefined metrics. Cache predictions for the galleries.

In [ ]:
records=[]; predictions={}; checkpoint_info=[]
for name,bands in EXPERIMENTS.items():
    ck=torch.load(RUN/name/"best.pt",map_location="cpu",weights_only=True)
    assert ck["config"]["architecture"]=="worldfloods_v1_unet_64_128_256_512_bilinear", "Select a Notebook 08 result directory."
    assert ck["bands"]==bands, "Checkpoint band order does not match."
    model=UNet(len(bands),2).to(device)
    model.load_state_dict(ck["model"]);model.eval()
    mean=np.array(ck["config"]["channel_mean"],dtype="float32")[:,None,None]
    std=np.array(ck["config"]["channel_std"],dtype="float32")[:,None,None]
    use_amp=bool(ck["config"]["amp"]) and device.type=="cuda"
    checkpoint_info.append(dict(experiment=name,epoch=ck["epoch"],bands=bands,config=ck["config"]))
    print(name,"best epoch:",ck["epoch"],"bands:",bands)
    for split,rows in SPLITS.items():
        for index,row in enumerate(rows):
            raw,y,_=read_sample(row)
            x=np.nan_to_num((raw-mean)/std)[:len(bands)]
            with torch.inference_mode(),torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):
                logits=model(torch.from_numpy(x.copy())[None].to(device))
            pred=logits.argmax(1)[0].cpu().numpy().astype("int8");pred[y==-1]=-1
            predictions[name,split,index]=pred
            counts=pixel_counts(pred,y);tn,fp,fn,tp=map(int,counts)
            records.append(dict(experiment=name,split=split,index=index,sample=row[0].replace("_S1Hand","_S2Hand"),
                event=row[0].split("_")[0],best_epoch=ck["epoch"],tn=tn,fp=fp,fn=fn,tp=tp,
                valid_pixels=int(counts.sum()),**scores(counts)))
        print(name,split,"completed:",len(rows))
    del model,ck
save_csv(OUT/"per_image_metrics.csv",records)
(OUT/"analysis_config.json").write_text(json.dumps(dict(run=str(RUN),device=str(device),
    selection="fixed test6 bolivia3; additional bottom3 NIR IoU per split",checkpoints=checkpoint_info),indent=2),encoding="utf-8")

## 4. Overall and event-group metrics

Pool confusion counts for global metrics and average defined image IoUs for mean IoU. Event groups follow dataset filename prefixes. Compare recalculated metrics with the selected run's original CSV; this is a descriptive single-run analysis.

In [ ]:
def aggregate(rows):
    counts=np.array([[r[k] for k in ["tn","fp","fn","tp"]] for r in rows]).sum(axis=0)
    vals=[r["water_iou"] for r in rows if np.isfinite(r["water_iou"])]
    out=scores(counts);out["global_water_iou"]=out.pop("water_iou")
    out.update(mean_water_iou=float(np.mean(vals)) if vals else float("nan"),
               images=len(rows),valid_images=sum(r["valid_pixels"]>0 for r in rows),
               images_with_water_union=len(vals),valid_pixels=int(counts.sum()))
    return out
summary=[];events=[]
for name in EXPERIMENTS:
    for split in SPLITS:
        rows=[r for r in records if r["experiment"]==name and r["split"]==split]
        summary.append(dict(experiment=name,split=split,best_epoch=rows[0]["best_epoch"],**aggregate(rows)))
        for event in sorted({r["event"] for r in rows}):
            events.append(dict(experiment=name,split=split,event=event,
                          **aggregate([r for r in rows if r["event"]==event])))
cols=["experiment","split","best_epoch","mean_water_iou","global_water_iou","f1","precision","recall"]
table(summary,cols)
table(events,["experiment","split","event","images","mean_water_iou","global_water_iou"])
save_csv(OUT/"summary_metrics.csv",summary);save_csv(OUT/"event_metrics.csv",events)
# Compare with the existing 08 export without overwriting it.
with (RUN/"test_metrics.csv").open(newline="",encoding="utf-8-sig") as f: original=list(csv.DictReader(f))
checks=[]
for row in summary:
    prior=next(r for r in original if r["experiment"]==row["experiment"] and r["split"]==row["split"])
    checks.append(dict(experiment=row["experiment"],split=row["split"],
        delta_mean_iou=row["mean_water_iou"]-float(prior["mean_water_iou"]),
        delta_global_iou=row["global_water_iou"]-float(prior["global_water_iou"])))
table(checks,["experiment","split","delta_mean_iou","delta_global_iou"])
save_csv(OUT/"comparison_to_08.csv",checks)
print("Compare recalculated metrics with the original run; check run, device and data if they differ.")
fig,axes=plt.subplots(1,2,figsize=(10,4),layout="constrained")
for ax,split in zip(axes,SPLITS):
    keys=["mean_water_iou","global_water_iou","f1"];x=np.arange(len(keys))
    for j,(name,color) in enumerate(zip(EXPERIMENTS,["#167caf","#df893b"])):
        r=next(r for r in summary if r["experiment"]==name and r["split"]==split)
        ax.bar(x+(j-.5)*.34,[r[k] for k in keys],width=.34,label=name,color=color)
    ax.set(xticks=x,xticklabels=["Mean IoU","Global IoU","F1"],ylim=(0,1),title=split)
    ax.legend();ax.grid(axis="y",alpha=.2)
for ext in ["png","pdf"]:fig.savefig(OUT/f"metrics_comparison.{ext}",dpi=150)
plt.show();plt.close(fig)

## 5. Predictions and spatial errors

Show the fixed-index gallery and a separate set of low-IoU error cases. Use the same colors in both models: beige = true non-water, blue = true water, orange = false positive, purple = false negative, grey = ignored.

Save selection indices in `gallery_selection.csv` and georeferenced prediction masks for the displayed images.

In [ ]:
mask_cmap=ListedColormap(["#d7dee7","#eee8d5","#167caf"])
mask_norm=BoundaryNorm([-1.5,-.5,.5,1.5],3)
error_colors=["#eee8d5","#167caf","#ed963b","#984ea3","#d7dee7"]
def error_map(pred,y):
    e=np.zeros_like(y,dtype="uint8")
    e[(y==1)&(pred==1)]=1;e[(y==0)&(pred==1)]=2;e[(y==1)&(pred==0)]=3;e[y==-1]=4
    return e
selection=[]
for split,rows in SPLITS.items():
    count=6 if split=="test" else 3
    for index in np.unique(np.linspace(0,len(rows)-1,min(count,len(rows)),dtype=int)):
        selection.append(dict(split=split,index=int(index),selection="Fixed index"))
    worst=sorted((r for r in records if r["experiment"]=="rgb_nir" and r["split"]==split
                  and np.isfinite(r["water_iou"])),key=lambda r:(r["water_iou"],r["index"]))[:3]
    for r in worst:selection.append(dict(split=split,index=r["index"],selection="Error case (bottom NIR IoU)"))
save_csv(OUT/"gallery_selection.csv",selection)
for item in selection:
    split,index=item["split"],item["index"];row=SPLITS[split][index]
    raw,y,profile=read_sample(row); rgb=np.moveaxis(raw[:3],0,-1)
    finite=np.isfinite(rgb).all(axis=-1)
    if finite.any():
        lo,hi=np.percentile(rgb[finite],[2,98],axis=0)
        rgb=np.clip((rgb-lo)/np.maximum(hi-lo,1e-6),0,1)
    rgb=np.nan_to_num(rgb)
    p3=predictions["rgb",split,index];p4=predictions["rgb_nir",split,index]
    fig,axes=plt.subplots(1,6,figsize=(18,3.7),layout="constrained")
    axes[0].imshow(rgb);axes[0].set_title("RGB")
    for ax,arr,title in zip(axes[1:4],[y,p3,p4],["Label","RGB prediction","RGB+NIR prediction"]):
        ax.imshow(arr,cmap=mask_cmap,norm=mask_norm,interpolation="nearest");ax.set_title(title)
    for ax,pred,title in zip(axes[4:],[p3,p4],["RGB errors","RGB+NIR errors"]):
        ax.imshow(error_map(pred,y),cmap=ListedColormap(error_colors),vmin=0,vmax=4,interpolation="nearest")
        ax.set_title(title)
    for ax in axes:ax.set_xticks([]);ax.set_yticks([])
    fig.legend(handles=[Patch(color=c,label=l) for c,l in zip(error_colors,["TN","TP","FP","FN","Ignored"])],
               loc="outside lower center",ncol=5)
    fig.suptitle(f"{item['selection']} | {split} #{index} | {row[0].replace('_S1Hand','_S2Hand')}",fontsize=10)
    stem=f"{split}_{index:03d}_"+("fixed" if item["selection"]=="Fixed index" else "error")
    fig.savefig(OUT/f"{stem}.png",dpi=150);plt.show();plt.close(fig)
    for name,pred in [("rgb",p3),("rgb_nir",p4)]:
        with rasterio.open(OUT/f"{split}_{index:03d}_{name}_mask.tif","w",
            **{**profile,"count":1,"dtype":"int16","nodata":-1,"compress":"deflate"}) as dst:
            dst.write(pred.astype("int16"),1)

## 6. Paired image differences

Subtract RGB IoU from RGB+NIR IoU for images where both are defined. Report the number of paired and excluded images, and the numbers with higher or lower NIR IoU. Image pairs are not assumed to be independent flood events.

In [ ]:
lookup={(r["experiment"],r["split"],r["index"]):r for r in records}
paired=[];lines=[]
for split,rows in SPLITS.items():
    delta=[]
    for index in range(len(rows)):
        a=lookup["rgb",split,index]["water_iou"];b=lookup["rgb_nir",split,index]["water_iou"]
        d=b-a if np.isfinite(a) and np.isfinite(b) else float("nan")
        paired.append(dict(split=split,index=index,rgb_iou=a,rgb_nir_iou=b,delta_iou=d))
        if np.isfinite(d):delta.append(d)
    lines.append(f"- {split}: {len(delta)} paired images; {len(rows)-len(delta)} images without a defined pair; "
                 f"higher NIR IoU in {sum(d>0 for d in delta)} images and lower in {sum(d<0 for d in delta)} images.")
    a=next(r for r in summary if r["experiment"]=="rgb" and r["split"]==split)
    b=next(r for r in summary if r["experiment"]=="rgb_nir" and r["split"]==split)
    lines.append(f"- {split}: difference between model-specific mean IoUs = {b['mean_water_iou']-a['mean_water_iou']:.4f}；"
                 f"This may differ from the mean over valid image pairs only.")
save_csv(OUT/"paired_image_differences.csv",paired)
report="\n".join(["# Fixed-model result analysis",f"RUN: {RUN.name}",*lines,
    "Descriptive results from the selected single-seed run.",
    "NYC application and ground-observation comparison are documented in Notebooks 11 and 12."])
(OUT/"results_summary.md").write_text(report,encoding="utf-8")
display(Markdown(report));print("All outputs:",OUT)

## Outputs

Save the executed notebook and its timestamped analysis directory. The model checkpoints and original training logs remain in the selected run. Continue with [11 · NYC inference](11_nyc_l1c_inference.ipynb).